# Agent 5 — Symptoms & Mood Agent (Package Edition)

Post-extraction demo for Agent 5. All class and function definitions now live in the `immunosense.agents.symptoms_mood` package.

**Special role:** Agent 5 is the canonical source of flare events. It produces `daily_flare_score()` that the Conductor distributes to Agents 1-4 as their `observe_flare()` input.

**Architecture (unchanged):**
- Layer 1: Disease-stratified norms + real NHANES PHQ-8 baseline
- Layer 2: `process_symptom_day(patient_id, date, disease, transcript, form_data, ...)`
- Layer 3: Per-patient adaptation with **pre-registered biological lags** for trigger detection

**Outputs:**
- `daily_flare_score()`: 0-1 score for Conductor distribution
- `jepa_embedding()`: 36-dim dense vector for the World Model
- `raw_hypothesis_evidence()`: ALL 10 hypotheses including sub-FDR for Conductor cross-corroboration

**Validations performed:**
1. Live Claude API smoke test (hand-crafted SLE patient transcript)
2. Concurrent fatigue trigger (positive control)
3. Predictive sleep trigger (positive control)
4. Reactive PHQ-8 trigger (bidirectional case - may be BH-suppressed, recovered via raw evidence)
5. Negative control (must produce zero patterns)


In [ ]:
# === Package imports ===
import os
import random as random_mod
from datetime import datetime

import numpy as np
import pandas as pd

from immunosense.agents.symptoms_mood import (
    SymptomsMoodAgent,
    process_symptom_day,
    compute_daily_flare_score,
    compute_jepa_embedding,
    CompositeSymptomSource,
    MockSymptomSource,
    VoiceTranscriptSource,
    StructuredFormSource,
    FlareButtonSource,
    DailySymptomMoodSummary,
    FetchedSymptoms,
    StubMemoryStore,
)
from immunosense.agents.symptoms_mood.types import (
    SYMPTOM_FEATURES, MOOD_FEATURES, ALL_FEATURES,
)

print('Imports OK')
print(f'SymptomsMoodAgent.agent_id      = {SymptomsMoodAgent.agent_id}')
print(f'SymptomsMoodAgent.output_dim    = {SymptomsMoodAgent.output_dim}')
print(f'SymptomsMoodAgent.agent_version = {SymptomsMoodAgent.agent_version}')
print()
print(f'Symptom features ({len(SYMPTOM_FEATURES)}): {SYMPTOM_FEATURES}')
print(f'Mood features ({len(MOOD_FEATURES)}):    {MOOD_FEATURES}')


In [ ]:
# === Section 1: Optional NHANES PHQ-8 baseline update ===
#
# Downloads NHANES P_DPQ.XPT (real US population data, ~8,000 adults) and
# overrides the literature-derived 'Mixed' PHQ-8 baseline with real values.
# Comment this cell out if you don't want network access.
try:
    from immunosense.agents.symptoms_mood.nhanes import update_disease_norms_with_nhanes
    stats = update_disease_norms_with_nhanes(verbose=True)
    if 'error' in stats:
        print(f'\n(NHANES not loaded - falling back to literature values for Mixed/PHQ-8)')
except Exception as e:
    print(f'NHANES skipped: {e}')


In [ ]:
# === Section 2: Live Claude API smoke test ===
#
# Hand-crafted SLE patient transcript -> Claude Haiku NLP extraction
# -> structured FetchedSymptoms -> DailySymptomMoodSummary.
# Skips gracefully if ANTHROPIC_API_KEY is not set.
SAMPLE_TRANSCRIPT = '''Today's been really tough. Woke up exhausted even after
eight hours of sleep - must have been restless. Knees feel about a 7 today,
hands a bit worse. Can't concentrate on anything, brain's just foggy. Skin's
been itchy near my elbows where the rash was last week. Skipped lunch, no
appetite. Pressed through work but barely. Mood is low, anxious about
tomorrow's meeting.'''

print('SAMPLE TRANSCRIPT:')
print(SAMPLE_TRANSCRIPT)
print()

try:
    import anthropic
    _ANTHROPIC_AVAILABLE = True
except ImportError:
    _ANTHROPIC_AVAILABLE = False

if not _ANTHROPIC_AVAILABLE:
    print('SKIPPED: anthropic SDK not installed. Run: pip install anthropic')
elif not os.environ.get('ANTHROPIC_API_KEY'):
    print('SKIPPED: ANTHROPIC_API_KEY not set in .env')
else:
    composite = CompositeSymptomSource(voice=VoiceTranscriptSource())
    today = datetime.now().strftime('%Y-%m-%d')

    summary = process_symptom_day(
        patient_id='smoke_test_001',
        target_date=today,
        disease='SLE',
        transcript=SAMPLE_TRANSCRIPT,
        composite_source=composite,
    )

    print(f'Date:    {summary.date}')
    print(f'Patient: {summary.patient_id}')
    print(f'Disease: {summary.disease}')
    print()
    print('Extracted symptoms (from voice transcript via Claude):')
    for feat in SYMPTOM_FEATURES:
        v = getattr(summary, feat)
        pct = summary.percentiles.get(feat)
        if v is not None and pct is not None:
            print(f'  {feat:25s}: {v}/10  (pct={pct:.0%})')
    print()
    print('Mood:')
    print(f'  PHQ-8: {summary.phq8_score}    (alert={summary.threshold_alerts.get("phq8")})')
    print(f'  GAD-7: {summary.gad7_score}    (alert={summary.threshold_alerts.get("gad7")})')
    print()
    print(f'NLP enrichment:')
    print(f'  emotional_valence:    {summary.emotional_valence}')
    print(f'  new_symptom_mentions: {summary.new_symptom_mentions}')
    print()
    print(f'DAILY FLARE SCORE (-> Conductor): {summary.flare_score:.3f}')
    print(f'Overall confidence: {summary.overall_confidence:.0%}')
    print()
    print('JEPA embedding (first 12 dims of 36):')
    emb = compute_jepa_embedding(summary)
    print(f'  {emb[:12]}')
    if summary.errors:
        print(f'\nErrors: {summary.errors}')


In [ ]:
# === Section 3: Layer 3 Validation - Synthetic Patient class ===
#
# Defines a synthetic patient with one configured trigger feature. Three
# trigger modes: predictive (lag>0), concurrent (lag=+1, +2), reactive (lag<0).
class SyntheticSymptomPatient:
    """Generates daily symptom records with one configured trigger."""

    BASELINE_DISTS = {
        'fatigue':             (4.5, 1.8),
        'joint_pain':          (4.0, 1.8),
        'brain_fog_severity':  (3.5, 1.7),
        'gi_distress':         (2.5, 1.5),
        'skin_severity':       (2.0, 1.3),
        'sleep_severity':      (4.0, 1.8),
        'energy_severity':     (4.0, 1.7),
        'wellness_severity':   (4.0, 1.6),
    }
    MOOD_DISTS = {
        'phq8_score': (7.5, 4.5),
        'gad7_score': (6.0, 4.0),
    }
    TRIGGER_THRESHOLDS = {
        'fatigue': 7.0, 'joint_pain': 7.0, 'sleep_severity': 7.0,
        'brain_fog_severity': 6.5, 'gi_distress': 6.0, 'skin_severity': 5.5,
        'energy_severity': 6.5, 'wellness_severity': 7.0,
    }
    REACTIVE_BOOST = 5.0

    def __init__(self, trigger_feature=None, trigger_lag_days=2,
                 trigger_flare_probability=0.85,
                 baseline_flare_probability=0.10, random_seed=42):
        self.trigger_feature = trigger_feature
        self.trigger_lag_days = trigger_lag_days
        self.trigger_flare_probability = trigger_flare_probability
        self.baseline_flare_probability = baseline_flare_probability
        self.rng = random_mod.Random(random_seed)
        self.np_rng = np.random.RandomState(random_seed)

    def _is_trigger_active(self, features):
        if self.trigger_feature is None or self.trigger_lag_days < 0:
            return False
        t = self.TRIGGER_THRESHOLDS.get(self.trigger_feature)
        v = features.get(self.trigger_feature)
        return t is not None and v is not None and v > t

    def generate(self, n_days=60, start_date='2026-03-01'):
        records, scheduled_flares = [], {}
        start = pd.Timestamp(start_date)

        for i in range(n_days):
            date = (start + pd.Timedelta(days=i)).strftime('%Y-%m-%d')
            features = {'date': date}
            for feat, (mean, std) in self.BASELINE_DISTS.items():
                v = self.np_rng.normal(mean, std)
                features[feat] = max(0.0, min(10.0, v))
            if pd.Timestamp(date).day_of_week == 0:
                for feat, (mean, std) in self.MOOD_DISTS.items():
                    cap = 24 if feat == 'phq8_score' else 21
                    features[feat] = max(0.0, min(cap, self.np_rng.normal(mean, std)))
            else:
                if records and 'phq8_score' in records[-1]:
                    features['phq8_score'] = records[-1]['phq8_score']
                    features['gad7_score'] = records[-1]['gad7_score']

            if self.trigger_lag_days >= 0:
                flare_prob = (self.trigger_flare_probability
                              if self._is_trigger_active(features)
                              else self.baseline_flare_probability)
            else:
                flare_prob = self.baseline_flare_probability

            if self.rng.random() < flare_prob:
                if self.trigger_lag_days >= 0:
                    flare_idx = i + self.trigger_lag_days
                else:
                    flare_idx = i
                if 0 <= flare_idx < n_days:
                    flare_date = (start + pd.Timedelta(days=flare_idx)).strftime('%Y-%m-%d')
                    severity = float(self.np_rng.uniform(0.5, 0.9))
                    scheduled_flares[flare_date] = max(scheduled_flares.get(flare_date, 0.0), severity)

            records.append(features)

        # Reactive mode: boost trigger feature on day (flare_day + |lag|)
        if self.trigger_lag_days < 0 and self.trigger_feature is not None:
            offset = abs(self.trigger_lag_days)
            for flare_date_str, severity in scheduled_flares.items():
                flare_day = (pd.Timestamp(flare_date_str) - start).days
                target_day = flare_day + offset
                if 0 <= target_day < n_days and target_day < len(records):
                    boost = self.REACTIVE_BOOST * severity
                    current = records[target_day].get(self.trigger_feature, 0.0)
                    if self.trigger_feature in self.MOOD_DISTS:
                        cap = 24 if self.trigger_feature == 'phq8_score' else 21
                    else:
                        cap = 10.0
                    records[target_day][self.trigger_feature] = min(cap, current + boost)

        return records, scheduled_flares


def _record_to_summary(record, flare_score=0.0):
    """Convert SyntheticSymptomPatient record to DailySymptomMoodSummary."""
    return DailySymptomMoodSummary(
        date=record['date'],
        patient_id='synthetic',
        disease='Mixed',
        fatigue=record.get('fatigue'),
        joint_pain=record.get('joint_pain'),
        brain_fog_severity=record.get('brain_fog_severity'),
        gi_distress=record.get('gi_distress'),
        skin_severity=record.get('skin_severity'),
        sleep_severity=record.get('sleep_severity'),
        energy_severity=record.get('energy_severity'),
        wellness_severity=record.get('wellness_severity'),
        phq8_score=record.get('phq8_score'),
        gad7_score=record.get('gad7_score'),
        flare_score=flare_score,
        overall_confidence=1.0,
    )


def run_validation_test(label, patient, expectation):
    records, flares = patient.generate(n_days=60)
    agent = SymptomsMoodAgent(n_permutations=500, patient_id='synthetic')

    for r in records:
        summary = _record_to_summary(r, flare_score=flares.get(r['date'], 0.0))
        agent.observe(summary)

    report = agent.analyze()
    print(f'\n{label}')
    print(f'  Expectation:  {expectation}')
    print(f'  Observed:     {report.n_days_observed} days, {report.n_flare_events} flare events')
    print(f'  Hypotheses:   {report.n_hypotheses_tested}')
    print(f'  Patterns ({len(report.detected_patterns)} found):')
    for p in report.detected_patterns:
        print(f'    {p.feature:30s} lag={p.lag_days:+d}d  effect={p.effect_size:+.3f}'
              f'  p={p.p_value:.3f}  q={p.q_value:.3f}  [{p.confidence}]')

    # Show raw evidence for Conductor consumption
    evidence = agent.raw_hypothesis_evidence()
    sub_threshold = [e for e in evidence if not e.survives_fdr and e.raw_p_value < 0.05]
    if sub_threshold:
        print(f'  Sub-threshold evidence (raw_p < 0.05, for Conductor cross-corroboration):')
        for e in sub_threshold:
            print(f'    {e.feature:25s} lag={e.lag_days:+d}d  effect={e.effect_size:+.3f}'
                  f'  raw_p={e.raw_p_value:.3f}  q={e.q_value:.3f}  [{e.biology_category}]')


print('Synthetic patient + validation helper defined.')


In [ ]:
# === Section 4: Run the four validation tests ===
print('=' * 70)
print('Agent 5 Layer 3 validation - 4 tests')
print('=' * 70)

run_validation_test(
    'TEST 1: Concurrent (fatigue trigger, lag=+2)',
    SyntheticSymptomPatient(trigger_feature='fatigue', trigger_lag_days=2, random_seed=42),
    'fatigue (>p85) at lag=+2d  [concurrent]',
)
run_validation_test(
    'TEST 2: Predictive (sleep trigger, lag=+2)',
    SyntheticSymptomPatient(trigger_feature='sleep_severity', trigger_lag_days=2, random_seed=43),
    'sleep_severity (>p85) at lag=+2d  [predictive]',
)
run_validation_test(
    'TEST 3: Reactive (phq8 trigger, lag=-3, true bidirectional)',
    SyntheticSymptomPatient(trigger_feature='phq8_score', trigger_lag_days=-3, random_seed=44),
    'phq8_score (>p85) at lag=-3d  [reactive - may be BH-suppressed]',
)
run_validation_test(
    'TEST 4: Negative control',
    SyntheticSymptomPatient(trigger_feature=None, random_seed=45),
    'Zero patterns at [high]; ideally zero patterns at all',
)

print('\n' + '=' * 70)
print('Validation complete.')
print('=' * 70)
